# YouTube Content Processor & Video Script Generator

Process markdown files from the news aggregator and use LLM to create detailed video scripts ready for production.

## Section 1: Import Required Libraries

Import libraries for file handling, LLM API, markdown parsing, and video script generation.

In [ ]:
import anthropic
import markdown
import json
import re
import textwrap
from pathlib import Path
from typing import Dict, List, Tuple
from datetime import datetime

## Section 2: Read and Parse Markdown Files

Load and parse the markdown files generated by the content generator to extract titles, tags, descriptions, and other elements.

In [ ]:
def read_latest_content_file(folder_path: str = "videos") -> Tuple[str, Dict]:
    """
    Read the latest generated content markdown file from the videos folder.
    Returns the file path and parsed content.
    """
    videos_dir = Path(folder_path)
    if not videos_dir.exists():
        raise FileNotFoundError(f"Videos folder not found: {folder_path}")
    
    # Find latest date folder
    date_folders = sorted([d for d in videos_dir.iterdir() if d.is_dir()])
    if not date_folders:
        raise FileNotFoundError("No date folders found in videos folder")
    
    latest_folder = date_folders[-1]
    
    # Find markdown files
    md_files = list(latest_folder.glob("*.md"))
    if not md_files:
        raise FileNotFoundError(f"No markdown files found in {latest_folder}")
    
    # Use Vietnamese file if available, otherwise English
    vi_file = [f for f in md_files if "-vi.md" in f.name]
    md_file = vi_file[0] if vi_file else md_files[0]
    
    with open(md_file, "r", encoding="utf-8") as f:
        content = f.read()
    
    return str(md_file), content

def parse_markdown_content(content: str) -> Dict:
    """
    Parse the markdown content and extract key sections.
    """
    parsed = {
        "titles": [],
        "tags": [],
        "description": "",
        "thumbnail_concept": "",
        "hook": "",
        "raw_content": content
    }
    
    # Extract titles
    title_section = re.search(r"### 🎯 TIÊU ĐỀ ĐỀ XUẤT.*?\n(.*?)(?=\n---|\n###|$)", content, re.DOTALL)
    if title_section:
        titles = re.findall(r"\*\*Lựa chọn \d+:\*\*\s*(.*)", title_section.group(1))
        parsed["titles"] = titles
    
    # Extract tags
    tags_section = re.search(r"### 🏷️ TAGS.*?`([^`]+)`", content, re.DOTALL)
    if tags_section:
        parsed["tags"] = [t.strip() for t in tags_section.group(1).split(",")]
    
    # Extract description
    desc_section = re.search(r"### 📝 MÔ TẢ VIDEO.*?\n(.*?)(?=\n---|\n### |$)", content, re.DOTALL)
    if desc_section:
        parsed["description"] = desc_section.group(1).strip()
    
    # Extract thumbnail concept
    thumb_section = re.search(r"### 💡 GỢI Ý NỘI DUNG THUMBNAIL:.*?\n(.*?)(?=\n###|$)", content, re.DOTALL)
    if thumb_section:
        parsed["thumbnail_concept"] = thumb_section.group(1).strip()
    
    # Extract hook
    hook_section = re.search(r"### 📊 HOOK MỞ ĐẦU.*?\n\"(.*?)\"", content, re.DOTALL)
    if hook_section:
        parsed["hook"] = hook_section.group(1).strip()
    
    return parsed

# Test loading and parsing
file_path, content = read_latest_content_file()
print(f"✅ Loaded file: {file_path}\n")

parsed_content = parse_markdown_content(content)
print(f"📌 Parsed Content:")
print(f"  Titles: {len(parsed_content['titles'])} found")
print(f"  Tags: {len(parsed_content['tags'])} found")
print(f"  Hook: {parsed_content['hook'][:100]}..." if parsed_content['hook'] else "  Hook: Not found")

## Section 3: Process Content with LLM

Use Claude API to enhance and expand the content into a detailed, engaging video narrative with better pacing and storytelling.

In [ ]:
# Initialize Anthropic client (uses ANTHROPIC_API_KEY from environment)
client = anthropic.Anthropic()

def enhance_content_with_llm(parsed_content: Dict) -> str:
    """
    Use Claude API to enhance and expand the content into a detailed video narrative.
    """
    
    # Build the prompt
    prompt = f"""
    You are a professional YouTube video scriptwriter. You have the following content framework:
    
    **Selected Title:** {parsed_content['titles'][0] if parsed_content['titles'] else 'Market Update'}
    
    **Current Hook:**
    {parsed_content['hook']}
    
    **Current Description:**
    {parsed_content['description']}
    
    **Task:** Create a detailed, engaging video script based on this framework. The script should:
    1. Have an engaging opening hook (15 seconds)
    2. Include clear transitions between segments
    3. Build narrative tension throughout
    4. End with a strong call-to-action
    5. Be approximately 15-20 minutes of speaking time
    6. Include timing cues for each section
    
    Format the output as:
    [TIME: MM:SS] - [SEGMENT TITLE]
    [NARRATION TEXT]
    
    Make it professional, engaging, and suitable for YouTube.
    """
    
    message = client.messages.create(
        model="claude-3-5-sonnet-20241022",
        max_tokens=2000,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    
    return message.content[0].text

# Process content with LLM
print("🤖 Processing content with Claude API...\n")
try:
    enhanced_script = enhance_content_with_llm(parsed_content)
    print("✅ LLM Enhancement Complete!")
    print("\n" + "="*50)
    print(enhanced_script[:500] + "..." if len(enhanced_script) > 500 else enhanced_script)
    print("="*50)
except Exception as e:
    print(f"⚠️ LLM Processing Error: {e}")
    print("Continuing with original content...")
    enhanced_script = parsed_content["hook"] + "\n\n" + parsed_content["description"]

In [ ]:
# Initialize Anthropic client (uses ANTHROPIC_API_KEY from environment)
client = anthropic.Anthropic()

def enhance_content_with_llm(parsed_content: Dict) -> str:
    """
    Use Claude API to enhance and expand the content into a detailed video narrative.
    """
    
    # Build the prompt
    prompt = f"""
    You are a professional YouTube video scriptwriter. You have the following content framework:
    
    **Selected Title:** {parsed_content['titles'][0] if parsed_content['titles'] else 'Market Update'}
    
    **Current Hook:**
    {parsed_content['hook']}
    
    **Current Description:**
    {parsed_content['description']}
    
    **Task:** Create a detailed, engaging video script based on this framework. The script should:
    1. Have an engaging opening hook (15 seconds)
    2. Include clear transitions between segments
    3. Build narrative tension throughout
    4. End with a strong call-to-action
    5. Be approximately 15-20 minutes of speaking time
    6. Include timing cues for each section
    
    Format the output as:
    [TIME: MM:SS] - [SEGMENT TITLE]
    [NARRATION TEXT]
    
    Make it professional, engaging, and suitable for YouTube.
    """
    
    message = client.messages.create(
        model="claude-3-5-sonnet-20241022",
        max_tokens=2000,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    
    return message.content[0].text

# Process content with LLM
print("🤖 Processing content with Claude API...\n")
try:
    enhanced_script = enhance_content_with_llm(parsed_content)
    print("✅ LLM Enhancement Complete!")
    print("\n" + "="*50)
    print(enhanced_script[:500] + "..." if len(enhanced_script) > 500 else enhanced_script)
    print("="*50)
except Exception as e:
    print(f"⚠️ LLM Processing Error: {e}")
    print("Continuing with original content...")
    enhanced_script = parsed_content["hook"] + "\n\n" + parsed_content["description"]

## Section 4: Generate Video Script

Transform the LLM output into a properly formatted video production script with timing, visuals, and narration.

In [ ]:
def generate_video_script(title: str, hook: str, enhanced_content: str, tags: List[str], thumbnail_concept: str) -> Dict:
    """
    Create a complete video production script.
    """
    
    script = {
        "title": title,
        "duration_minutes": 18,
        "tags": tags,
        "thumbnail_concept": thumbnail_concept,
        "sections": []
    }
    
    # Opening hook section
    script["sections"].append({
        "time_start": "00:00",
        "time_end": "00:15",
        "title": "Opening Hook",
        "type": "narration",
        "content": hook,
        "visual_notes": "Hook - zoom in on speaker, eye contact with camera",
        "bgm": "Intense background music, low volume"
    })
    
    # Main content section
    script["sections"].append({
        "time_start": "00:15",
        "time_end": "17:30",
        "title": "Main Content",
        "type": "mixed",
        "content": enhanced_content,
        "visual_notes": "Mix of: Screen sharing, charts, news headlines, speaker reactions",
        "bgm": "Dynamic background music matching market mood"
    })
    
    # Call to action section
    script["sections"].append({
        "time_start": "17:30",
        "time_end": "18:00",
        "title": "Call to Action",
        "type": "narration",
        "content": "If you found this analysis valuable, please like, subscribe, and hit the notification bell for daily market updates. Drop your thoughts in the comments below!",
        "visual_notes": "Show subscribe button animation, highlight comment section",
        "bgm": "Upbeat background music"
    })
    
    return script

# Generate the full video script
video_script = generate_video_script(
    title=parsed_content['titles'][0] if parsed_content['titles'] else "Market Update",
    hook=parsed_content['hook'],
    enhanced_content=enhanced_script,
    tags=parsed_content['tags'],
    thumbnail_concept=parsed_content['thumbnail_concept']
)

print("🎬 Video Script Generated!")
print(f"\n📹 Video Details:")
print(f"  Title: {video_script['title']}")
print(f"  Duration: {video_script['duration_minutes']} minutes")
print(f"  Sections: {len(video_script['sections'])}")
print(f"  Tags: {len(video_script['tags'])} tags")

## Section 5: Format Output for Video Production

Export the script in formats compatible with video creation tools (Premiere Pro, DaVinci Resolve, CapCut, etc.) and save as JSON and formatted text.

In [ ]:
def export_video_script(script: Dict, output_folder: str = "videos_scripts") -> Tuple[str, str]:
    """
    Export the video script in multiple formats for video production tools.
    Returns paths to JSON and formatted text files.
    """
    
    # Create output folder
    output_path = Path(output_folder)
    output_path.mkdir(exist_ok=True)
    
    # Generate filename from date and title
    timestamp = datetime.now().strftime("%Y-%m-%d")
    safe_title = "".join(c for c in script["title"] if c.isalnum() or c in " -_")[:50]
    
    # Export as JSON (for programmatic use)
    json_file = output_path / f"script_{timestamp}_{safe_title}.json"
    with open(json_file, "w", encoding="utf-8") as f:
        json.dump(script, f, ensure_ascii=False, indent=2)
    
    # Export as formatted text (for reading/editing)
    text_file = output_path / f"script_{timestamp}_{safe_title}.txt"
    text_content = f"""
╔══════════════════════════════════════════════════════════════════════╗
║                     YOUTUBE VIDEO PRODUCTION SCRIPT                  ║
╚══════════════════════════════════════════════════════════════════════╝

📺 TITLE: {script['title']}
⏱️  DURATION: {script['duration_minutes']} minutes
📍 DATE: {timestamp}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📌 VIDEO METADATA
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🏷️  TAGS ({len(script['tags'])}):
{', '.join(script['tags'][:10])}

🎨 THUMBNAIL CONCEPT:
{script['thumbnail_concept']}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🎬 SCRIPT SECTIONS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

"""
    
    for i, section in enumerate(script["sections"], 1):
        text_content += f"\n{'='*70}\n"
        text_content += f"SECTION {i}: {section['title'].upper()}\n"
        text_content += f"{'='*70}\n"
        text_content += f"⏱️  Time: {section['time_start']} → {section['time_end']}\n"
        text_content += f"🎭 Type: {section['type']}\n"
        text_content += f"🎵 BGM: {section.get('bgm', 'Not specified')}\n\n"
        
        text_content += f"📝 NARRATION/CONTENT:\n"
        text_content += f"{textwrap.fill(section['content'], width=70)}\n\n"
        
        text_content += f"👀 VISUAL NOTES:\n"
        text_content += f"{textwrap.fill(section['visual_notes'], width=70)}\n"
    
    text_content += f"\n\n{'='*70}\n"
    text_content += "END OF SCRIPT\n"
    text_content += f"{'='*70}\n"
    
    with open(text_file, "w", encoding="utf-8") as f:
        f.write(text_content)
    
    return str(json_file), str(text_file)

# Export the script
json_path, text_path = export_video_script(video_script)

print(f"✅ Script Exported Successfully!\n")
print(f"📂 Output Files:")
print(f"  JSON: {json_path}")
print(f"  Text: {text_path}\n")

# Show preview of text format
print("📄 Preview of Text Format:")
print("=" * 70)
with open(text_path, "r", encoding="utf-8") as f:
    preview = f.read()
    lines = preview.split("\n")
    print("\n".join(lines[:30]))
    if len(lines) > 30:
        print("\n... [rest of script] ...\n")